# H6b: Residual 7x Directional Advantage Across Activation Functions

## Scientific Context

In experiment H6, after carefully controlling for learning-rate artifacts, we discovered that Muon retains
a **genuine ~7x loss advantage** over SGD in deep linear networks when both optimizers are evaluated at
their respective optimal learning rates. This was a surprise: the raw advantage initially appeared much
larger, but most of it was attributable to Muon tolerating higher effective LRs. The residual 7x factor,
however, is real and demands explanation.

**The core question of H6b:** Is this 7x advantage a peculiarity of linear networks, or does it persist
(and potentially change magnitude) when we introduce nonlinear activation functions?

## Why This Matters

The Muon optimizer replaces raw gradients with their Newton-Schulz orthogonalized version -- effectively
projecting each gradient matrix onto the nearest orthogonal matrix. In a deep linear network, this
amounts to equalizing the singular values of the gradient update, which directly combats the spectral
imbalance that makes deep linear nets hard to optimize.

But nonlinear activations introduce fundamentally different geometry:
- **ReLU**: Creates sparse, piecewise-linear regions. Gradients become sparse. Does SV equalization still help when half the gradient entries are zero?
- **tanh**: Saturating activation causes vanishing gradients in deep nets. Muon's orthogonalization might rescue signal from near-zero gradient matrices -- potentially making the advantage *larger*.
- **GELU**: Smooth approximation to ReLU. Less sparsity, but still nonlinear. An intermediate test case.

## Experimental Design

For each activation in {linear, ReLU, tanh, GELU}:
1. **LR sweep phase**: Test 10 LR candidates per optimizer (3 seeds, 500 steps) to find each optimizer's best LR for that activation.
2. **Full evaluation phase**: Train with the best LR across 10 seeds, record final loss.
3. **Compute advantage**: ratio = SGD_best_loss / Muon_best_loss.

## Hypothesis Tests

| Test | Question | Criterion |
|------|----------|-----------|
| **T1** | Does Muon beat SGD for ALL activations? | advantage > 1.0x for every activation |
| **T2** | Is the advantage consistent across activations? | all advantages in [0.5x, 20x] range |
| **T3** | Does tanh show larger advantage than ReLU? | advantage(tanh) > advantage(ReLU) |

T3 is the most theoretically interesting: if Muon's benefit comes from SV equalization rescuing
vanishing-gradient layers, then tanh (which suffers most from gradient vanishing) should show the
largest Muon advantage.

## 1. Imports and Environment

In [ ]:
"""
H6b: Residual 7x Directional Advantage Across Activations
============================================================

MOTIVATION (from H6 surprise):
  After correcting for LR artifacts, Muon retains a genuine 7x advantage
  over SGD in deep linear nets. This was measured at optimal LR for both.
  QUESTION: Does this 7x advantage persist, grow, or shrink across
  activation functions? If the advantage changes dramatically with
  nonlinearity, the mechanism may be geometry-specific, not universal.

PROTOCOL:
  For each activation in {linear, ReLU, tanh, GELU}:
    For each optimizer in {SGD, Muon}:
      Sweep LR (10 candidates each), then train 500 steps.
    Compute advantage = SGD_best_loss / Muon_best_loss.

KEY TESTS:
  T1: Does Muon beat SGD (at optimal LR) for ALL activations?
  T2: Is the advantage consistent (within 0.5-20x across activations)?
  T3: Does tanh (vanishing gradients) show larger Muon advantage than ReLU?
      (Hypothesis: SV equalization helps more when gradients vanish.)

Setup: 4-layer, 32x32, 500 steps, 10 seeds, LR swept per activation.
"""
print(__doc__)

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

## 2. Hyperparameters

The architecture is deliberately modest: 4 layers of 32x32 matrices, trained for 500 steps.
This is small enough that we can sweep learning rates exhaustively (10 candidates per optimizer)
across 4 activation functions without prohibitive cost, yet deep enough that the spectral
dynamics of deep networks are active.

Key design choices:
- **Momentum = 0.9**: Standard heavy-ball momentum. Both optimizers use the same momentum coefficient,
  so any difference is purely from the gradient preprocessing (Newton-Schulz vs. identity).
- **NS_ITERS = 5**: Newton-Schulz iterations for Muon's orthogonalization. Five iterations typically
  bring the iterate very close to the true polar factor.
- **10 seeds**: Enough for stable mean estimates; we report mean over finite-loss seeds.
- **BATCH_SIZE = 64**: Each "batch" is a fixed random matrix pair (X, Y). We are fitting a
  fixed mapping, not sampling from a distribution -- so this is a deterministic optimization problem.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 10
BATCH_SIZE = 64

print("=== Experiment Hyperparameters ===")
print(f"  Network dimensions:  {DIM} x {DIM}")
print(f"  Network depth:       {NUM_LAYERS} layers")
print(f"  Training steps:      {NUM_STEPS}")
print(f"  Momentum:            {MOMENTUM}")
print(f"  Newton-Schulz iters: {NS_ITERS}")
print(f"  Seeds for evaluation:{NUM_SEEDS}")
print(f"  Batch size:          {BATCH_SIZE}")
print(f"  Total params:        {NUM_LAYERS * DIM * DIM} ({NUM_LAYERS} x {DIM}x{DIM} weight matrices)")

## 3. Learning Rate Grids

A critical methodological point: **we use separate LR grids for Muon and SGD.** This is essential
because Muon's Newton-Schulz orthogonalization normalizes the gradient to have spectral norm ~1,
so effective step sizes differ dramatically between the two optimizers.

The SGD grid spans [0.0005, 0.2] while the Muon grid spans [0.001, 0.05]. These ranges were
chosen based on preliminary experiments to bracket the optimal LR for each optimizer in 32x32
deep networks. Using the same grid for both would be a confound -- exactly the artifact we
corrected for in H6.

In [ ]:
MUON_LRS = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]
SGD_LRS = [0.2, 0.1, 0.05, 0.03, 0.02, 0.01, 0.005, 0.003, 0.001, 0.0005]

print("=== Learning Rate Grids ===")
print(f"  Muon LR candidates ({len(MUON_LRS)}): {MUON_LRS}")
print(f"    Range: [{min(MUON_LRS)}, {max(MUON_LRS)}], ratio max/min = {max(MUON_LRS)/min(MUON_LRS):.0f}x")
print(f"  SGD  LR candidates ({len(SGD_LRS)}):  {SGD_LRS}")
print(f"    Range: [{min(SGD_LRS)}, {max(SGD_LRS)}], ratio max/min = {max(SGD_LRS)/min(SGD_LRS):.0f}x")
print(f"\n  Note: Muon LRs are ~{max(SGD_LRS)/max(MUON_LRS):.0f}x smaller than SGD LRs at the top end.")
print(f"  This reflects Newton-Schulz normalization making Muon's effective step inherently larger.")

## 4. Activation Functions Under Test

We test four activations spanning a range of geometric properties:

| Activation | Key Property | Expected Effect on Muon Advantage |
|------------|-------------|-----------------------------------|
| **Linear** | Baseline (identity) | ~7x (replicate H6 result) |
| **ReLU** | Sparse gradients (50% zeros) | Possibly reduced -- sparsity may limit SV equalization benefit |
| **tanh** | Saturating, vanishing gradients | Possibly increased -- Muon may rescue near-zero gradient matrices |
| **GELU** | Smooth, non-sparse | Intermediate -- less sparsity than ReLU, less saturation than tanh |

The linear case serves as our internal control: we should reproduce the ~7x advantage from H6.

In [ ]:
ACTIVATIONS = ['linear', 'relu', 'tanh', 'gelu']

print(f"Activations to test: {ACTIVATIONS}")
print(f"Total experiment configurations: {len(ACTIVATIONS)} activations x 2 optimizers x {NUM_SEEDS} seeds = {len(ACTIVATIONS)*2*NUM_SEEDS} full training runs")
print(f"Plus LR sweep: {len(ACTIVATIONS)} activations x 2 optimizers x {len(MUON_LRS)} LRs x 3 seeds = {len(ACTIVATIONS)*2*len(MUON_LRS)*3} sweep runs")

## 5. Newton-Schulz Orthogonalization

The Newton-Schulz iteration is the heart of Muon. Given a matrix M, it computes the polar factor
U = M (M^T M)^{-1/2}, which is the closest orthogonal matrix to M in Frobenius norm.

The iteration is: X_{k+1} = (3/2) X_k - (1/2) X_k (X_k^T X_k)

This converges quadratically to U when the initial iterate X_0 = M / ||M||_F has singular values
in (0, sqrt(3)). After convergence, the output has all singular values equal to 1, which means
the update direction is "spectrally democratic" -- no singular direction is privileged over another.

**Why this matters for the experiment**: When gradients become ill-conditioned (as happens with
vanishing gradients in tanh nets or sparse gradients in ReLU nets), Newton-Schulz rescues the
small singular values by inflating them to 1. This is the mechanism we hypothesize gives Muon
its advantage, and why we expect the advantage to be *largest* for tanh.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

## 6. Activation Functions and Their Derivatives

We implement four activation functions and their element-wise derivatives for manual backpropagation.

**GELU implementation note**: We use the tanh-based approximation
`GELU(x) = 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))`,
which matches PyTorch's default GELU to high precision. The derivative is computed analytically
from this closed-form expression rather than numerically, ensuring exact gradients.

**Why manual backprop?** We implement forward and backward passes by hand (no autograd framework)
to have full control over the computation and to verify that any optimizer differences are genuine,
not artifacts of framework-level optimizations.

In [ ]:
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

In [ ]:
def gelu_deriv(x):
    s = np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)
    t = np.tanh(s)
    ds = np.sqrt(2 / np.pi) * (1 + 3 * 0.044715 * x**2)
    return 0.5 * (1 + t) + 0.5 * x * (1 - t**2) * ds

In [ ]:
def apply_act(x, act_name):
    if act_name == 'linear':
        return x
    elif act_name == 'relu':
        return np.maximum(0, x)
    elif act_name == 'tanh':
        return np.tanh(x)
    elif act_name == 'gelu':
        return gelu(x)

In [ ]:
def apply_act_deriv(pre, act_name):
    if act_name == 'linear':
        return np.ones_like(pre)
    elif act_name == 'relu':
        return (pre > 0).astype(float)
    elif act_name == 'tanh':
        return 1 - np.tanh(pre)**2
    elif act_name == 'gelu':
        return gelu_deriv(pre)

## 7. Weight Initialization

Weights are initialized as `I + 0.1 * N(0,1)` -- a small perturbation from identity. This is
important for several reasons:

1. **Near-identity start**: The product W_4 W_3 W_2 W_1 starts near identity, so the initial
   loss is moderate (not exploding or vanishing).
2. **Same initialization for both optimizers**: Both SGD and Muon start from identical weights
   for each seed, ensuring a fair comparison.
3. **Seed offset**: Weight seeds are `s + 5000` while data seeds are `s`, preventing any
   correlation between weight initialization and target structure.

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]

## 8. Forward Pass, Loss, and Gradient Computation

The network computes: `output = W_4 * act(W_3 * act(W_2 * act(W_1 * X)))`.

Note that the **last layer has no activation** -- this is standard practice (the output is a
linear readout). Activations are applied only at intermediate layers (layers 1 through L-1).

The loss is MSE: `L = 0.5 * mean(sum((pred - Y)^2, axis=0))`.

Gradients are computed via manual backpropagation with the chain rule. For each layer l, the
gradient is `dL/dW_l = delta_l @ activations_{l-1}^T`, where delta propagates backward through
the network, picking up activation derivatives at each intermediate layer.

In [ ]:
def forward(weights, X, act):
    pre_acts = []
    out = X.copy()
    for idx, W in enumerate(weights):
        pre = W @ out
        pre_acts.append(pre)
        if idx < len(weights) - 1:
            out = apply_act(pre, act)
        else:
            out = pre
    return out, pre_acts

In [ ]:
def compute_loss(weights, X, Y, act):
    pred, _ = forward(weights, X, act)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y, act):
    L = len(weights)
    N = X.shape[1]
    acts_post = [X.copy()]
    pre_acts = []
    out = X.copy()
    for idx, W in enumerate(weights):
        pre = W @ out
        pre_acts.append(pre)
        if idx < L - 1:
            out = apply_act(pre, act)
        else:
            out = pre
        acts_post.append(out)
    delta = (acts_post[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts_post[l].T
        if l > 0:
            delta = weights[l].T @ delta
            delta = delta * apply_act_deriv(pre_acts[l - 1], act)
    return grads

## 9. Training Loop

The training loop is where the two optimizers diverge:

**SGD (with momentum):**
```
m_t = beta * m_{t-1} + grad
W = W - lr * m_t
```

**Muon (with momentum):**
```
m_t = beta * m_{t-1} + NewtonSchulz(grad)
W = W - lr * m_t
```

The ONLY difference is that Muon passes the gradient through Newton-Schulz before adding it to
the momentum buffer. This orthogonalizes the update direction, equalizing all singular values to 1.

**Divergence detection**: If loss exceeds 1e10 or becomes non-finite, we return infinity. This
handles cases where the learning rate is too large for a given activation (ReLU and GELU networks
can be more sensitive to LR than linear networks).

In [ ]:
def train(weights_init, X, Y, lr, optimizer, act):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y, act)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y, act)
        for i in range(len(weights)):
            if optimizer == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]
    return compute_loss(weights, X, Y, act)

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = rng.randn(DIM, BATCH_SIZE) * 0.3
    return X, Y

## 10. Main Experiment: LR Sweep + Full Evaluation

The experiment proceeds in two phases per activation function:

### Phase 1: Learning Rate Sweep
For each optimizer, we test all 10 LR candidates using 3 seeds and select the LR that gives
the lowest mean final loss. This ensures we compare each optimizer at its *best* LR for each
activation, not at a fixed LR that might favor one optimizer.

### Phase 2: Full Evaluation
With the best LR identified, we train across all 10 seeds and compute the mean final loss.
The advantage ratio is then: `advantage = SGD_best_loss / Muon_best_loss`.

An advantage > 1 means Muon achieves lower loss. An advantage of 7x means SGD's best loss is
7 times worse than Muon's best loss.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H6b: RESIDUAL 7x DIRECTIONAL ADVANTAGE ACROSS ACTIVATIONS")
    print("=" * 100)
    print(f"Activations: {ACTIVATIONS}")
    print(f"Network: {NUM_LAYERS}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"Seeds: {seeds}")
    print(f"LR sweep uses first 3 seeds: {seeds[:3]}")
    print()

    results = {}
    all_sweep_data = {}  # store sweep details for diagnostics

    for act_idx, act in enumerate(ACTIVATIONS):
        print(f"\n{'='*100}")
        print(f"  ACTIVATION {act_idx+1}/{len(ACTIVATIONS)}: {act.upper()}")
        print(f"{'='*100}")

        # --- Diagnostic: show what the activation does to a sample input ---
        sample_input = np.linspace(-2, 2, 9)
        sample_output = apply_act(sample_input, act)
        print(f"\n  Activation profile (sample inputs -2..+2):")
        print(f"    input:  {np.array2string(sample_input, precision=2, separator=', ')}")
        print(f"    output: {np.array2string(sample_output, precision=2, separator=', ')}")
        if act == 'tanh':
            print(f"    Note: outputs saturate near +/-1 -- gradients will vanish for |x| > 2")
        elif act == 'relu':
            print(f"    Note: {np.sum(sample_input <= 0)}/{len(sample_input)} inputs are zeroed out (negative region)")
        elif act == 'gelu':
            print(f"    Note: smooth approximation -- negative inputs get soft-gated rather than hard-zeroed")
        elif act == 'linear':
            print(f"    Note: identity function -- baseline for comparison with H6 results")

        # LR sweep (3 seeds, full NUM_STEPS steps)
        print(f"\n  --- Phase 1: Learning Rate Sweep ---")
        print(f"  Testing {len(MUON_LRS)} LRs per optimizer, {len(seeds[:3])} seeds each")
        best_lrs = {}
        for opt, candidates in [('muon', MUON_LRS), ('sgd', SGD_LRS)]:
            best_lr, best_loss = candidates[-1], float('inf')
            sweep_results = []
            for lr in candidates:
                losses = []
                for s in seeds[:3]:
                    X, Y = make_data(s)
                    w = init_weights(s + 5000)
                    fl = train(w, X, Y, lr, opt, act)
                    losses.append(fl)
                ml = np.mean([l for l in losses if np.isfinite(l)]) if any(np.isfinite(l) for l in losses) else float('inf')
                n_diverged = sum(1 for l in losses if not np.isfinite(l))
                sweep_results.append((lr, ml, n_diverged))
                if ml < best_loss:
                    best_loss = ml
                    best_lr = lr
            best_lrs[opt] = best_lr
            all_sweep_data[(act, opt)] = sweep_results

            # Print full sweep table
            print(f"\n    {opt.upper()} LR sweep results:")
            print(f"    {'LR':>8}  {'Mean Loss':>14}  {'Diverged':>8}  {'Status'}")
            print(f"    {'-'*50}")
            for lr, ml, nd in sweep_results:
                marker = " <-- BEST" if lr == best_lr else ""
                status = f"({nd}/3 diverged)" if nd > 0 else ""
                if np.isfinite(ml):
                    print(f"    {lr:>8.4f}  {ml:>14.6e}  {nd:>8}  {status}{marker}")
                else:
                    print(f"    {lr:>8.4f}  {'inf':>14}  {nd:>8}  ALL DIVERGED")
            print(f"    ==> Best LR for {opt.upper()}: {best_lr:.4f} (loss={best_loss:.6e})")

        # Report LR ratio
        lr_ratio = best_lrs['sgd'] / best_lrs['muon'] if best_lrs['muon'] > 0 else float('inf')
        print(f"\n  LR ratio (SGD/Muon): {lr_ratio:.1f}x")
        print(f"  This reflects Newton-Schulz spectral normalization effect on effective step size.")

        # Full training
        print(f"\n  --- Phase 2: Full Evaluation at Optimal LRs ---")
        print(f"  Training with {NUM_SEEDS} seeds, {NUM_STEPS} steps each")
        for opt in ['muon', 'sgd']:
            losses = []
            for s in seeds:
                X, Y = make_data(s)
                w = init_weights(s + 5000)
                fl = train(w, X, Y, best_lrs[opt], opt, act)
                losses.append(fl)
            finite = [l for l in losses if np.isfinite(l)]
            mean_l = np.mean(finite) if finite else float('inf')
            std_l = np.std(finite) if len(finite) > 1 else 0.0
            n_diverged = NUM_SEEDS - len(finite)
            results[(act, opt)] = {'lr': best_lrs[opt], 'mean_loss': mean_l}

            print(f"\n    {opt.upper()} (lr={best_lrs[opt]:.4f}):")
            print(f"      Individual seed losses: {[f'{l:.4e}' if np.isfinite(l) else 'inf' for l in losses]}")
            print(f"      Mean final loss: {mean_l:.6e}")
            print(f"      Std final loss:  {std_l:.6e}")
            print(f"      Coefficient of variation: {std_l/mean_l:.2%}" if mean_l > 0 else "")
            if n_diverged > 0:
                print(f"      WARNING: {n_diverged}/{NUM_SEEDS} seeds diverged!")
            else:
                print(f"      All {NUM_SEEDS} seeds converged successfully.")

        # Immediate advantage for this activation
        ml_muon = results[(act, 'muon')]['mean_loss']
        ml_sgd = results[(act, 'sgd')]['mean_loss']
        adv = ml_sgd / max(ml_muon, 1e-30) if np.isfinite(ml_sgd) and np.isfinite(ml_muon) else float('nan')
        print(f"\n  >>> ADVANTAGE for {act.upper()}: {adv:.2f}x (SGD loss / Muon loss)")
        if adv > 1:
            print(f"      Muon wins by {adv:.1f}x -- SGD's best loss is {adv:.1f} times worse.")
        elif adv < 1:
            print(f"      SGD wins by {1/adv:.1f}x -- unexpected! Muon is worse here.")
        else:
            print(f"      Dead heat -- optimizers perform identically.")

    # ========================================================================
    # Results table
    # ========================================================================
    print(f"\n\n{'=' * 100}")
    print("RESULTS: Muon vs SGD at Optimal LR per Activation")
    print(f"{'=' * 100}")

    print(f"\n  {'Activation':>10}  {'Muon loss':>12} {'(lr)':>8}  {'SGD loss':>12} {'(lr)':>8}  {'Advantage':>12}")
    print("  " + "-" * 70)

    advantages = {}
    for act in ACTIVATIONS:
        ml = results[(act, 'muon')]['mean_loss']
        sl = results[(act, 'sgd')]['mean_loss']
        adv = sl / max(ml, 1e-30) if np.isfinite(sl) and np.isfinite(ml) else float('nan')
        advantages[act] = adv
        print(f"  {act:>10}  {ml:>12.4e} {results[(act,'muon')]['lr']:>8.4f}  "
              f"{sl:>12.4e} {results[(act,'sgd')]['lr']:>8.4f}  {adv:>12.1f}x")

    # Summary statistics
    valid_advs = [v for v in advantages.values() if np.isfinite(v)]
    if valid_advs:
        print(f"\n  Summary across activations:")
        print(f"    Min advantage:  {min(valid_advs):.2f}x ({min(advantages, key=advantages.get)})")
        print(f"    Max advantage:  {max(valid_advs):.2f}x ({max(advantages, key=advantages.get)})")
        print(f"    Mean advantage: {np.mean(valid_advs):.2f}x")
        print(f"    Std advantage:  {np.std(valid_advs):.2f}x")
        print(f"    Range ratio (max/min): {max(valid_advs)/max(min(valid_advs), 1e-30):.2f}x")

    # ========================================================================
    # Hypothesis tests
    # ========================================================================
    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    valid_advs = [v for v in advantages.values() if np.isfinite(v)]
    t1 = all(v > 1.0 for v in valid_advs)
    t2 = all(0.5 < v < 20.0 for v in valid_advs) if valid_advs else False
    t3 = advantages.get('tanh', 0) > advantages.get('relu', float('inf'))

    print(f"\n  T1: Muon beats SGD for ALL activations? --> {'PASS' if t1 else 'FAIL'}")
    if t1:
        print(f"      All {len(valid_advs)} advantages are > 1.0x: {[f'{v:.1f}x' for v in valid_advs]}")
        print(f"      The directional advantage is universal across activation types.")
    else:
        failing = [act for act, v in advantages.items() if np.isfinite(v) and v <= 1.0]
        print(f"      FAIL: Activations where SGD >= Muon: {failing}")
        print(f"      The directional advantage is NOT universal -- activation-dependent.")

    print(f"\n  T2: Advantage consistent (0.5-20x range)? --> {'PASS' if t2 else 'FAIL'}")
    if t2:
        print(f"      All advantages fall in [0.5, 20.0]: range is well-bounded.")
        print(f"      This suggests a common mechanism (SV equalization) operating similarly")
        print(f"      across activation types, not an activation-specific artifact.")
    else:
        outliers = [f"{act}={v:.1f}x" for act, v in advantages.items() if np.isfinite(v) and not (0.5 < v < 20.0)]
        print(f"      FAIL: Out-of-range advantages: {outliers}")
        print(f"      The advantage magnitude is highly activation-dependent.")

    print(f"\n  T3: tanh advantage > ReLU advantage? --> {'PASS' if t3 else 'FAIL'}")
    print(f"       tanh={advantages.get('tanh', 'N/A'):.1f}x, ReLU={advantages.get('relu', 'N/A'):.1f}x")
    if t3:
        ratio = advantages.get('tanh', 0) / max(advantages.get('relu', 1), 1e-30)
        print(f"       tanh advantage is {ratio:.2f}x larger than ReLU advantage.")
        print(f"       Interpretation: Vanishing gradients (tanh) amplify Muon's benefit.")
        print(f"       This supports the hypothesis that SV equalization rescues signal from")
        print(f"       near-zero gradient matrices -- precisely what vanishing gradients produce.")
    else:
        print(f"       Interpretation: Either (a) ReLU sparsity is a bigger problem than tanh")
        print(f"       vanishing gradients, or (b) the advantage mechanism is not primarily")
        print(f"       about rescuing small singular values.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

## 11. Interpretation and Conclusions

### What the results mean for the "Muon as RG Gauge Fixing" theory

This experiment tests whether Muon's directional advantage (the residual ~7x factor from H6) is a
**universal geometric property** of Newton-Schulz orthogonalization or a **special case** limited to
linear networks.

**If T1 passes (Muon wins everywhere):** The advantage is activation-agnostic. Orthogonalizing the
gradient update direction is fundamentally beneficial regardless of the loss landscape's curvature
structure. This strongly supports interpreting Muon as a form of "gauge fixing" -- it removes
redundant degrees of freedom from the update direction that are universally harmful.

**If T2 passes (consistent magnitude):** The ~7x factor is not a coincidence of linear geometry.
The same mechanism operates with similar strength across very different activation landscapes.
This makes the advantage a robust feature of the optimizer, not a quirk.

**If T3 passes (tanh > ReLU advantage):** This is the most mechanistically informative result.
Tanh networks suffer from vanishing gradients, which manifest as gradient matrices with very
small singular values. Newton-Schulz inflates these to 1, restoring full directional information.
ReLU networks instead have sparse (but not small) gradients. Finding that tanh benefits more
confirms that the key mechanism is **singular value equalization rescuing near-zero gradient
directions**, not just generic direction normalization.

### Implications for scaling

If the advantage is activation-universal, it should persist in practical architectures (which use
ReLU/GELU). If T3 holds, then architectures with more severe gradient pathologies (very deep nets,
attention layers with softmax saturation) may benefit even more from Muon-style orthogonalization.

### Follow-up experiments
- **H11**: Test if Muon's advantage interacts with dead neuron pathology (ReLU-specific).
- **H15**: Compare Muon's orthogonalization with explicit SVD clamping (more expensive but exact).
- **Depth scaling**: Does the advantage grow with depth? (Deeper = worse conditioning = more to fix.)